# CPSA 01 — build the half-bust YOLO-pose dataset

Inputs: the public COCO 2017 images dataset attached on Kaggle
(`awsaf49/coco-2017-dataset`, any tree with `train2017/` and `val2017/`).
The COCO-WholeBody v1.0 annotations are downloaded from the official Google
Drive links (https://github.com/jin-s13/COCO-WholeBody) into
`/tmp/coco_wholebody`; enable Internet for this notebook. If the download is
blocked, attach a dataset with the two `coco_wholebody_*_v1.0.json` files
instead: they are used when present.

Set `COMMIT_SHA` below to the exact repository commit for this run. Outputs go
to `/kaggle/working/halfbust62` and are re-published as a dataset version
before the session ends. Set `SCHEMA = "coco17"` for Option 1. No GPU needed.

In [ ]:
import subprocess
import sys

# Pin the exact CPSA_2026 commit for a reproducible Kaggle run. The [sources]
# extra installs gdown for the annotation download.
COMMIT_SHA = "042d7a556154d7e3aa5dd65b66629fad88904f6b"

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        f"cpsa-ml[sources] @ git+https://github.com/fcabecciaw/CPSA_2026@{COMMIT_SHA}#subdirectory=ml",
    ],
    check=True,
)

In [ ]:
from pathlib import Path

from cpsa_ml.datasets.sources import find_coco_root, wholebody_annotations

INPUT_DIR = Path("/kaggle/input")  # attached datasets
ANNOTATION_CACHE = Path("/tmp/coco_wholebody")  # outside /kaggle/working

COCO_ROOT = find_coco_root(INPUT_DIR)
WHOLEBODY = wholebody_annotations(INPUT_DIR, ANNOTATION_CACHE)
print("COCO root:", COCO_ROOT)
for split, path in WHOLEBODY.items():
    print(split, path)

In [ ]:
from cpsa_ml.datasets.halfbust import BuildConfig, build_halfbust_dataset

OUT = Path("/kaggle/working/halfbust62")
SCHEMA = "halfbust62"
OCCLUSION_P = 0.3
SEED = 0

for split in ("train", "val"):
    report = build_halfbust_dataset(
        BuildConfig(
            coco_root=COCO_ROOT,
            wholebody_json=WHOLEBODY[split],
            split=split,
            schema=SCHEMA,
            out=OUT,
            occlusion_p=OCCLUSION_P,
            seed=SEED,
        )
    )
    print(split, report.to_dict())

In [ ]:
print((OUT / "data.yaml").read_text())
print((OUT / "stats.json").read_text())

## Next steps

Re-publish `/kaggle/working/halfbust62` as a new version of the
`fabiocabeccia/cpsa-halfbust` dataset, then run notebook 03 (FP32 training) or
notebook 04 (variant study) with the new version attached.